## Example 1: Create the azure.common exception classes

In [ ]:
from azure.common import AzureException, AzureHttpError, AzureConflictHttpError, AzureMissingResourceHttpError

# Example 1: Azure Common Exception Classes
print("Example 1: Azure Common Exception Classes")

try:
    # Demonstrate Azure exception hierarchy
    print("✓ Azure exception classes imported successfully")

    # Show exception hierarchy
    print(f"AzureException: {AzureException}")
    print(f"AzureHttpError: {AzureHttpError}")
    print(f"AzureConflictHttpError: {AzureConflictHttpError}")
    print(f"AzureMissingResourceHttpError: {AzureMissingResourceHttpError}")

    # Test exception creation
    base_exception = AzureException("Base Azure exception")
    http_exception = AzureHttpError("HTTP error occurred", 404)
    conflict_exception = AzureConflictHttpError("Resource conflict", 409)
    missing_exception = AzureMissingResourceHttpError("Resource not found", 404)

    print(f"✓ Created AzureException: {base_exception}")
    print(f"✓ Created AzureHttpError: {http_exception}")
    print(f"✓ Created AzureConflictHttpError: {conflict_exception}")
    print(f"✓ Created AzureMissingResourceHttpError: {missing_exception}")

    # Test inheritance
    print(f"✓ AzureHttpError is subclass of AzureException: {issubclass(AzureHttpError, AzureException)}")
    print(f"✓ AzureConflictHttpError is subclass of AzureHttpError: {issubclass(AzureConflictHttpError, AzureHttpError)}")

except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"azure_common example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

print("Example 1 completed\n")

## Example 2: Look up API profiles with KnownProfiles

In [ ]:
from azure.profiles import KnownProfiles, ProfileDefinition

print("Example 2: KnownProfiles / ProfileDefinition")

names = [p.name for p in KnownProfiles]
print(f"Known profiles: {names}")
assert "default" in names
assert "latest" in names

latest = KnownProfiles.latest
assert isinstance(latest.value, ProfileDefinition)
print(f"latest profile label: {latest.value.label}")

hybrid = KnownProfiles.v2020_09_01_hybrid.value
print(f"hybrid profile label: {hybrid.label}, entries: {len(hybrid.get_profile_dict())}")
assert isinstance(hybrid.label, str)
assert len(hybrid.get_profile_dict()) > 0

assert KnownProfiles.from_name("latest") is KnownProfiles.latest
print("KnownProfiles.from_name round-trip OK")

print("Example 2 completed\n")

## Example 3: Raise and catch through the exception hierarchy

In [ ]:
from azure.common import AzureException, AzureHttpError, AzureConflictHttpError, AzureMissingResourceHttpError

print("Example 3: exception hierarchy raise/catch round-trip")

try:
    raise AzureConflictHttpError("resource already exists", 409)
except AzureHttpError as err:
    print(f"caught as AzureHttpError: {err} (status_code={err.status_code})")
    assert err.status_code == 409

try:
    raise AzureMissingResourceHttpError("no such resource", 404)
except AzureException as err:
    assert isinstance(err, AzureHttpError)
    assert err.status_code == 404
    print(f"caught as AzureException: status_code={err.status_code}")

assert issubclass(AzureConflictHttpError, AzureHttpError)
assert issubclass(AzureMissingResourceHttpError, AzureHttpError)
assert issubclass(AzureHttpError, AzureException)
print("hierarchy invariants hold")

print("Example 3 completed\n")

## Example 4: Cloud and client-factory helpers

In [ ]:
import importlib
import importlib.util

from azure.common import cloud

# Example 4: Cloud and client-factory helpers
# get_cli_active_cloud() reads the active cloud from an installed azure-cli-core older than 2.21;
# in any other setup it raises ImportError that names that requirement.
assert callable(cloud.get_cli_active_cloud)
try:
    active_cloud = cloud.get_cli_active_cloud()
except ImportError as err:
    assert "azure-cli-core" in str(err), err
    print("get_cli_active_cloud() -> ImportError (needs azure-cli-core < 2.21)")
else:
    assert isinstance(active_cloud.name, str)
    print("active cloud:", active_cloud.name)

# client_factory builds management clients from an auth file, a JSON dict or the CLI profile.
# It depends on the optional msrestazure package and refuses to import without it.
if importlib.util.find_spec("msrestazure") is not None:
    from azure.common import client_factory

    for name in ("get_client_from_auth_file", "get_client_from_json_dict", "get_client_from_cli_profile"):
        assert callable(getattr(client_factory, name)), name
    print("client_factory helpers are available")
else:
    try:
        importlib.import_module("azure.common.client_factory")
    except ImportError as err:
        assert "msrestazure" in str(err), err
        print("azure.common.client_factory -> ImportError (needs msrestazure)")
    else:
        raise AssertionError("azure.common.client_factory imported although msrestazure is not installed")